# **IS510 Presenttaion - Types Predictions**

***By Team 2, Gold Cohort, MSIS 2026***

## **Step 0. Preparation**

In [1]:
# Install Gradio
!pip -q install gradio

# Import Packages
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss, accuracy_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import ClassifierChain, MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

# Upload the datasets to my Github and get the links
AllPokemons_url = "https://raw.githubusercontent.com/FlalaGoGoGo/IS510_Cases/refs/heads/main/IS510_Pokemon_List.csv"

# Read CSV files into DataFrames
AllPokemons_df = pd.read_csv(AllPokemons_url)

In [2]:
# Get dataset shape
print(AllPokemons_df.shape)

(1215, 30)


In [3]:
# Preview training data
display(AllPokemons_df.head())

,dexnum,name,generation,type1,type2,species,height,weight,ability1,ability2,...,base_friendship,base_exp,growth_rate,egg_group1,egg_group2,percent_male,percent_female,egg_cycles,special_group,url
0,1.0,Bulbasaur,1.0,Grass,Poison,Seed Pokémon,0.7,6.9,Overgrow,Chlorophyll,...,50,64,Medium Slow,Grass,Monster,87.5,12.5,20,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...
1,2.0,Ivysaur,1.0,Grass,Poison,Seed Pokémon,1.0,13.0,Overgrow,Chlorophyll,...,50,142,Medium Slow,Grass,Monster,87.5,12.5,20,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...
2,3.0,Venusaur,1.0,Grass,Poison,Seed Pokémon,2.0,100.0,Overgrow,Chlorophyll,...,50,236,Medium Slow,Grass,Monster,87.5,12.5,20,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...
3,4.0,Charmander,1.0,Fire,NaN,Lizard Pokémon,0.6,8.5,Blaze,Solar Power,...,50,62,Medium Slow,Dragon,Monster,87.5,12.5,20,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...
4,5.0,Charmeleon,1.0,Fire,NaN,Flame Pokémon,1.1,19.0,Blaze,Solar Power,...,50,142,Medium Slow,Dragon,Monster,87.5,12.5,20,Ordinary,https://img.pokemondb.net/sprites/scarlet-viol...


In [4]:
# Get column names
print(AllPokemons_df.columns.tolist())

['dexnum', 'name', 'generation', 'type1', 'type2', 'species', 'height', 'weight', 'ability1', 'ability2', 'hidden_ability', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'total', 'ev_yield', 'catch_rate', 'base_friendship', 'base_exp', 'growth_rate', 'egg_group1', 'egg_group2', 'percent_male', 'percent_female', 'egg_cycles', 'special_group', 'url']


In [5]:
# Check missing values in each dataset
print(AllPokemons_df.isna().sum())

dexnum             190
name               190
generation         190
type1              190
type2              689
species            190
height             190
weight             190
ability1           190
ability2           357
hidden_ability     685
hp                 190
attack             190
defense            190
sp_atk             190
sp_def             190
speed              190
total              190
ev_yield           190
catch_rate         190
base_friendship    190
base_exp           190
growth_rate        190
egg_group1         190
egg_group2         936
percent_male       345
percent_female     345
egg_cycles         190
special_group      190
url                  0
dtype: int64


## **Step 1: Prepare Features and Multi-Label Target**

In [6]:
# Use df1 as the working copy so we can use df2, df3, etc. later
df1 = AllPokemons_df.copy()

# Fill missing secondary types; some Pokémon only have one type
df1['type2'] = df1['type2'].fillna('None')

df1[['type1', 'type2']] = df1[['type1', 'type2']].astype(str)
df1[['type1', 'type2']] = df1[['type1', 'type2']].replace({'nan': 'None', 'NaN': 'None'})

# Encode primary and secondary types into a multi-label binary matrix
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df1[['type1', 'type2']].values.tolist())

# Convert percentage columns to numeric, setting non-numeric entries to NaN
for col in ['percent_male', 'percent_female']:
    df1[col] = pd.to_numeric(df1[col], errors='coerce')

# Convert string-based numeric columns (possibly containing '—') to floats
for col in ['egg_cycles', 'base_friendship', 'base_exp']:
    df1[col] = (
        df1[col].astype(str)
        .str.replace('—', '', regex=False)
        .replace('', np.nan)
        .astype(float)
    )

## **Step 2: Define Preprocessing Pipeline**

In [7]:
# Numeric features used for modeling
numeric_features = [
    'generation', 'height', 'weight', 'hp', 'attack', 'defense',
    'sp_atk', 'sp_def', 'speed', 'total', 'catch_rate',
    'percent_male', 'percent_female',
    'egg_cycles', 'base_friendship', 'base_exp'
]

# Categorical features with manageable cardinality
categorical_features = ['growth_rate', 'egg_group1', 'egg_group2', 'special_group']

# Preprocessing pipeline for numeric features: impute then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing pipeline for categorical features: impute then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing for numeric and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## **Step 3: Split Data and Apply Preprocessing**

In [8]:
# Split the data into training and testing sets (no stratification because label combinations are many and sparse)
X_train, X_test, y_train, y_test = train_test_split(
    df1, y, test_size=0.2, random_state=42
)

# Fit the preprocessor on the training data and transform both train and test sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## **Step 4: Define and Evaluate Multiple Models**

In [9]:
def evaluate_model(name, model, use_processed=False):
    if use_processed:
        model.fit(X_train_processed, y_train)
        y_pred = model.predict(X_test_processed)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    micro_f1 = f1_score(y_test, y_pred, average='micro')
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    hamming = hamming_loss(y_test, y_pred)
    exact_match = accuracy_score(y_test, y_pred)

    print(f"{name}:")
    print(f"  Micro F1 score:        {micro_f1:.4f}")
    print(f"  Macro F1 score:        {macro_f1:.4f}")
    print(f"  Hamming loss:          {hamming:.4f}")
    print(f"  Exact match accuracy:  {exact_match:.4f}")
    print("-" * 50)

    return (micro_f1, macro_f1, hamming, exact_match)

# Dictionary to store results
results = {}

# 1. Logistic Regression (One-Vs-Rest)
ovr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', OneVsRestClassifier(LogisticRegression(max_iter=300)))
])
results['LogReg_OVR'] = evaluate_model("Logistic Regression (OVR)", ovr_model)

# 2. Logistic Regression Classifier Chain, C=1
chain_lr_c1 = ClassifierChain(
    base_estimator=LogisticRegression(max_iter=300, C=1.0),
    order='random', random_state=42
)
results['LogReg_Chain_C1'] = evaluate_model(
    "Logistic Regression Chain (C=1)",
    chain_lr_c1,
    use_processed=True
)

# 3. Logistic Regression Classifier Chain, C=10
chain_lr_c10 = ClassifierChain(
    base_estimator=LogisticRegression(max_iter=300, C=10.0),
    order='random', random_state=42
)
results['LogReg_Chain_C10'] = evaluate_model(
    "Logistic Regression Chain (C=10)",
    chain_lr_c10,
    use_processed=True
)

# 4. Random Forest MultiOutput
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', MultiOutputClassifier(RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    )))
])
results['RandomForest'] = evaluate_model("Random Forest MultiOutput", rf_model)

# 5. Extra Trees MultiOutput
extra_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', MultiOutputClassifier(ExtraTreesClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    )))
])
results['ExtraTrees'] = evaluate_model("Extra Trees MultiOutput", extra_model)

# 6. Gradient Boosting MultiOutput
gb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', MultiOutputClassifier(GradientBoostingClassifier(
        n_estimators=100, random_state=42
    )))
])
results['GradBoost'] = evaluate_model("Gradient Boosting MultiOutput", gb_model)

# Print a summary of all models
print("Summary of model performance (Micro F1, Macro F1, Hamming loss, Exact match):")
for model_name, metrics in results.items():
    print(f"{model_name}: {metrics}")

Logistic Regression (OVR):
  Micro F1 score:        0.5765
  Macro F1 score:        0.3881
  Hamming loss:          0.0671
  Exact match accuracy:  0.2510
--------------------------------------------------
Logistic Regression Chain (C=1):
  Micro F1 score:        0.6125
  Macro F1 score:        0.4353
  Hamming loss:          0.0682
  Exact match accuracy:  0.3333
--------------------------------------------------
Logistic Regression Chain (C=10):
  Micro F1 score:        0.6104
  Macro F1 score:        0.4819
  Hamming loss:          0.0760
  Exact match accuracy:  0.3909
--------------------------------------------------
Random Forest MultiOutput:
  Micro F1 score:        0.5627
  Macro F1 score:        0.3774
  Hamming loss:          0.0650
  Exact match accuracy:  0.2593
--------------------------------------------------
Extra Trees MultiOutput:
  Micro F1 score:        0.6176
  Macro F1 score:        0.4487
  Hamming loss:          0.0630
  Exact match accuracy:  0.3251
----------

## **Step 5: Select the Best Model (No Re-fitting)**

In [10]:
# Choose the best-performing model based on Micro F1 then Macro F1
best_model_name = max(
    results.items(),
    key=lambda kv: (kv[1][0], kv[1][1])  # prioritize micro F1 then macro F1
)[0]
print(f"Best model selected: {best_model_name}")

# For convenience, set final_model and X_test_for_final based on best_model_name
if best_model_name == 'LogReg_Chain_C10':
    final_model = chain_lr_c10
    X_test_for_final = X_test_processed
elif best_model_name == 'LogReg_Chain_C1':
    final_model = chain_lr_c1
    X_test_for_final = X_test_processed
else:
    final_model = {
        'LogReg_OVR': ovr_model,
        'RandomForest': rf_model,
        'ExtraTrees': extra_model,
        'GradBoost': gb_model
    }[best_model_name]
    X_test_for_final = X_test  # pipeline includes preprocessing

# Evaluate the final model (optional)
final_preds = final_model.predict(X_test_for_final)
final_micro_f1 = f1_score(y_test, final_preds, average='micro')
final_macro_f1 = f1_score(y_test, final_preds, average='macro')
print(f"Final micro F1: {final_micro_f1:.4f}")
print(f"Final macro F1: {final_macro_f1:.4f}")

Best model selected: ExtraTrees
Final micro F1: 0.6176
Final macro F1: 0.4487


## **Step 6: Primary / Secondary / Both Accuracy (threshold-based decoding)**

In [11]:
# Use predictions from the best classifier chain (C=10) with default threshold (0.5)
# chain_lr_c10 has been fitted in Step 4 using X_train_processed
y_pred_chain = chain_lr_c10.predict(X_test_processed)

classes = mlb.classes_
test_indices = X_test.index.to_list()

primary_correct = []
secondary_correct = []
secondary_mask = []  # track which samples truly have a second type
both_correct = []

for i, idx in enumerate(test_indices):
    # True primary and secondary types from df1
    true1 = df1.loc[idx, 'type1']
    true2 = df1.loc[idx, 'type2']

    # Predicted label set for this sample
    pred_vec = y_pred_chain[i]   # 0/1 vector
    pred_labels = set(classes[pred_vec == 1])

    # 1) Primary type accuracy: is the true primary type in the predicted label set?
    primary_correct.append(1 if true1 in pred_labels else 0)

    # 2) Secondary type accuracy: only for Pokémon that truly have a second type
    if true2 != 'None':
        secondary_mask.append(True)
        secondary_correct.append(1 if true2 in pred_labels else 0)
    else:
        secondary_mask.append(False)
        secondary_correct.append(None)

    # 3) Both types correct:
    #    build true set and predicted set (ignoring 'None')
    true_set = {true1}
    if true2 != 'None':
        true_set.add(true2)

    pred_set_wo_none = {t for t in pred_labels if t != 'None'}

    both_correct.append(1 if pred_set_wo_none == true_set else 0)

# Compute accuracies
primary_acc = np.mean(primary_correct)

sec_vals = [v for v, m in zip(secondary_correct, secondary_mask) if m]
secondary_acc = np.mean(sec_vals) if len(sec_vals) > 0 else np.nan

both_acc = np.mean(both_correct)

print(f"Primary type accuracy (set-based):   {primary_acc:.4f}")
print(f"Secondary type accuracy (set-based): {secondary_acc:.4f}")
print(f"Both types accuracy   (set-based):   {both_acc:.4f}")

Primary type accuracy (set-based):   0.5885
Secondary type accuracy (set-based): 0.4455
Both types accuracy   (set-based):   0.2675


## **Step 7: Primary / Secondary / Both Accuracy (Top-k probability decoding)**

In [12]:
# Get probability predictions from the best classifier chain model (C=10)
# chain_lr_c10 has been fitted in Step 4
y_proba_chain = chain_lr_c10.predict_proba(X_test_processed)

classes = mlb.classes_
test_indices = X_test.index.to_list()

primary_correct_topk = []
secondary_correct_topk = []
secondary_mask_topk = []  # track which samples truly have a second type
both_correct_topk = []

# Threshold for deciding whether to assign a second type
secondary_prob_threshold = 0.30

for i, idx in enumerate(test_indices):
    # True primary and secondary types from df1
    true1 = df1.loc[idx, 'type1']
    true2 = df1.loc[idx, 'type2']

    # Probability vector for this sample (one probability per type)
    proba_vec = np.array(y_proba_chain[i])

    # Sort types by predicted probability in descending order
    sorted_indices = np.argsort(-proba_vec)
    sorted_types = classes[sorted_indices]
    sorted_probs = proba_vec[sorted_indices]

    # 1) Predicted primary type: highest-probability class
    pred_primary = sorted_types[0]

    # 2) Predicted secondary type: default is 'None'
    pred_secondary = 'None'

    # Look for the second-best class that is not the same as primary
    # and whose probability exceeds the threshold and is not 'None'
    for t, p in zip(sorted_types[1:], sorted_probs[1:]):
        if t == pred_primary:
            continue
        if (t != 'None') and (p >= secondary_prob_threshold):
            pred_secondary = t
            break

    # 1) Primary type accuracy (top-k based)
    primary_correct_topk.append(1 if true1 == pred_primary else 0)

    # 2) Secondary type accuracy (only for Pokémon that truly have a second type)
    if true2 != 'None':
        secondary_mask_topk.append(True)
        secondary_correct_topk.append(1 if true2 == pred_secondary else 0)
    else:
        secondary_mask_topk.append(False)
        secondary_correct_topk.append(None)

    # 3) Both types correct:
    #    Construct true and predicted sets (ignoring 'None')
    true_set = {true1}
    if true2 != 'None':
        true_set.add(true2)

    pred_set = {pred_primary}
    if pred_secondary != 'None':
        pred_set.add(pred_secondary)

    both_correct_topk.append(1 if pred_set == true_set else 0)

# Compute top-k based accuracies
primary_acc_topk = np.mean(primary_correct_topk)

sec_vals_topk = [v for v, m in zip(secondary_correct_topk, secondary_mask_topk) if m]
secondary_acc_topk = np.mean(sec_vals_topk) if len(sec_vals_topk) > 0 else np.nan

both_acc_topk = np.mean(both_correct_topk)

print(f"Top-k primary type accuracy:   {primary_acc_topk:.4f}")
print(f"Top-k secondary type accuracy: {secondary_acc_topk:.4f}")
print(f"Top-k both types accuracy:     {both_acc_topk:.4f}")
print(f"(Secondary threshold = {secondary_prob_threshold})")

Top-k primary type accuracy:   0.4362
Top-k secondary type accuracy: 0.1909
Top-k both types accuracy:     0.3210
(Secondary threshold = 0.3)


In [13]:
# Step 7 (revised): Primary / Secondary / Both accuracy using improved top-k decoding

# Get probability predictions from the best classifier chain model (C=10)
# chain_lr_c10 has been fitted in Step 4
y_proba_chain = chain_lr_c10.predict_proba(X_test_processed)

classes = mlb.classes_
test_indices = X_test.index.to_list()

primary_correct_topk = []
secondary_correct_topk = []
secondary_mask_topk = []  # track which samples truly have a second type
both_correct_topk = []

# Threshold for deciding whether to assign a second type (you can tune this)
secondary_prob_threshold = 0.25  # try a bit lower than 0.30

for i, idx in enumerate(test_indices):
    # True primary and secondary types from df1
    true1 = df1.loc[idx, 'type1']
    true2 = df1.loc[idx, 'type2']

    # Probability vector for this sample (one probability per class)
    proba_vec = np.array(y_proba_chain[i])

    # --------- Choose primary type (exclude 'None') ---------
    # Indices of all classes except 'None'
    non_none_mask = (classes != 'None')
    non_none_probs = proba_vec[non_none_mask]
    non_none_types = classes[non_none_mask]

    # Primary = the non-'None' type with the highest probability
    primary_idx = np.argmax(non_none_probs)
    pred_primary = non_none_types[primary_idx]

    # --------- Choose secondary type (optional, exclude primary and 'None') ---------
    pred_secondary = 'None'

    # Candidates for secondary: non-'None' and not equal to pred_primary
    sec_candidates = []
    for t, p in zip(non_none_types, non_none_probs):
        if t == pred_primary:
            continue
        sec_candidates.append((t, p))

    # Sort candidates by probability in descending order
    sec_candidates.sort(key=lambda x: -x[1])

    # If we have any candidate, check the top one against threshold
    if sec_candidates:
        best_sec_type, best_sec_prob = sec_candidates[0]
        if best_sec_prob >= secondary_prob_threshold:
            pred_secondary = best_sec_type

    # 1) Primary type accuracy (top-k improved)
    primary_correct_topk.append(1 if true1 == pred_primary else 0)

    # 2) Secondary type accuracy (only for Pokémon that truly have a second type)
    if true2 != 'None':
        secondary_mask_topk.append(True)
        secondary_correct_topk.append(1 if true2 == pred_secondary else 0)
    else:
        secondary_mask_topk.append(False)
        secondary_correct_topk.append(None)

    # 3) Both types correct:
    #    Construct true and predicted sets (ignoring 'None')
    true_set = {true1}
    if true2 != 'None':
        true_set.add(true2)

    pred_set = {pred_primary}
    if pred_secondary != 'None':
        pred_set.add(pred_secondary)

    both_correct_topk.append(1 if pred_set == true_set else 0)

# Compute improved top-k based accuracies
primary_acc_topk = np.mean(primary_correct_topk)

sec_vals_topk = [v for v, m in zip(secondary_correct_topk, secondary_mask_topk) if m]
secondary_acc_topk = np.mean(sec_vals_topk) if len(sec_vals_topk) > 0 else np.nan

both_acc_topk = np.mean(both_correct_topk)

print(f"Improved top-k primary type accuracy: {primary_acc_topk:.4f}")
print(f"Improved top-k secondary type accuracy: {secondary_acc_topk:.4f}")
print(f"Improved top-k both types accuracy: {both_acc_topk:.4f}")
print(f"(Secondary threshold = {secondary_prob_threshold})")

Improved top-k primary type accuracy: 0.4239
Improved top-k secondary type accuracy: 0.2091
Improved top-k both types accuracy: 0.2469
(Secondary threshold = 0.25)


## **Step 8: Helper function to predict from Pokémon name**

In [14]:
def predict_types_from_name(pokemon_name):
    """
    Given a Pokémon name (exactly as in df1['name']), return detailed info:
    - True primary type
    - Predicted primary type
    - Whether primary is predicted correctly
    - True secondary type
    - Predicted secondary type
    - Whether secondary is predicted correctly
    - Whether both types are predicted correctly
    """

    # Find the row for this Pokémon
    row = df1[df1['name'] == pokemon_name]
    if row.empty:
        msg = f"Pokémon '{pokemon_name}' not found in the dataset."
        return ("N/A", "N/A", "N/A", "N/A", "N/A", "N/A", msg)

    # Take the first matching row
    row = row.iloc[0]

    # Build a one-row DataFrame; we can pass all columns, preprocessor will pick needed ones
    input_df = row.to_frame().T

    # True types
    true_primary = row['type1']
    true_secondary = row['type2']  # may be 'None'

    # Preprocess features
    X_processed = preprocessor.transform(input_df)

    # Get probability predictions from the trained classifier chain
    y_proba = chain_lr_c10.predict_proba(X_processed)[0]  # probabilities for this one sample
    classes = mlb.classes_
    proba_vec = np.array(y_proba)

    # --------- Improved top-k decoding (same逻辑 as Step 7) ---------
    # Exclude 'None' when choosing primary type
    non_none_mask = (classes != 'None')
    non_none_probs = proba_vec[non_none_mask]
    non_none_types = classes[non_none_mask]

    # Primary = the non-'None' type with the highest probability
    primary_idx = np.argmax(non_none_probs)
    pred_primary = non_none_types[primary_idx]

    # Secondary type: choose the best remaining type if probability >= threshold
    secondary_prob_threshold = 0.25  # keep consistent with Step 7
    pred_secondary = 'None'

    # Candidates for secondary: non-'None' and not equal to primary
    sec_candidates = []
    for t, p in zip(non_none_types, non_none_probs):
        if t == pred_primary:
            continue
        sec_candidates.append((t, p))

    # Sort candidates by probability in descending order
    sec_candidates.sort(key=lambda x: -x[1])

    # If we have any candidate, check top one against threshold
    if sec_candidates:
        best_sec_type, best_sec_prob = sec_candidates[0]
        if best_sec_prob >= secondary_prob_threshold:
            pred_secondary = best_sec_type

    # ----- correctness for primary -----
    primary_correct_bool = (pred_primary == true_primary)
    primary_correct_str = "✅ Correct" if primary_correct_bool else "❌ Wrong"

    # ----- correctness for secondary -----
    if true_secondary == 'None':
        # No real secondary type
        secondary_correct_str = "N/A (no secondary type)"
    else:
        secondary_correct_bool = (pred_secondary == true_secondary)
        secondary_correct_str = "✅ Correct" if secondary_correct_bool else "❌ Wrong"

    # ----- correctness for both types together -----
    true_set = {true_primary}
    if true_secondary != 'None':
        true_set.add(true_secondary)

    pred_set = {pred_primary}
    if pred_secondary != 'None':
        pred_set.add(pred_secondary)

    both_correct_bool = (pred_set == true_set)
    both_correct_str = "✅ Both correct" if both_correct_bool else "❌ Not both correct"

    # Ensure we always return 7 strings
    return (
        str(true_primary),
        str(pred_primary),
        primary_correct_str,
        str(true_secondary),
        str(pred_secondary),
        secondary_correct_str,
        both_correct_str
    )

In [15]:
print(predict_types_from_name("Charizard"))

('Fire', 'Fire', '✅ Correct', 'Flying', 'Grass', '❌ Wrong', '❌ Not both correct')


## **Step 9: Gradio interactive demo (dropdown with dex number + name)**

In [16]:
import gradio as gr
import requests
from PIL import Image
from io import BytesIO

# Make sure df1 exists (in case this cell is run standalone)
if "df1" not in globals():
    df1 = AllPokemons_df.copy()

# Ensure dexnum is numeric and handle missing values
df1["dexnum"] = pd.to_numeric(df1["dexnum"], errors="coerce")

# Helper function to download image from URL and return a PIL image
def fetch_image(url):
    if not isinstance(url, str) or url.strip() == "":
        return None
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        return Image.open(BytesIO(resp.content))
    except Exception:
        return None

# Build options like "001 - Bulbasaur", sorted by dexnum
pokemon_df_sorted = df1[df1["dexnum"].notna()].copy().sort_values("dexnum")
pokemon_df_sorted["dexnum"] = pokemon_df_sorted["dexnum"].astype(int)

pokemon_options = []
for _, row in pokemon_df_sorted.iterrows():
    num = row["dexnum"]
    name = row["name"]
    label = f"{num:03d} - {name}"   # e.g., "001 - Bulbasaur"
    pokemon_options.append(label)

# Build a mapping from Pokémon name to image URL
name_to_url = dict(zip(df1["name"], df1["url"]))

def predict_from_dropdown(option_label):

    if option_label is None or option_label == "":
        # First element is image (None), followed by 7 text fields
        return (
            None,
            "N/A", "N/A", "Please select a Pokémon.",
            "N/A", "N/A", "N/A", "N/A"
        )

    # Split "001 - Bulbasaur" -> "Bulbasaur"
    try:
        pokemon_name = option_label.split(" - ", 1)[1].strip()
    except IndexError:
        pokemon_name = option_label.strip()

    # Look up image URL and fetch the image
    image_url = name_to_url.get(pokemon_name, None)
    pokemon_img = fetch_image(image_url)

    # Call the existing helper to get type predictions
    (
        true_primary,
        pred_primary,
        primary_correct,
        true_secondary,
        pred_secondary,
        secondary_correct,
        both_correct
    ) = predict_types_from_name(pokemon_name)

    # Return image + all text outputs
    return (
        pokemon_img,
        true_primary,
        pred_primary,
        primary_correct,
        true_secondary,
        pred_secondary,
        secondary_correct,
        both_correct
    )

# Define Gradio interface
demo = gr.Interface(
    fn = predict_from_dropdown,
    inputs = gr.Dropdown(choices=pokemon_options, label="Choose a Pokémon"),
    outputs = [
        gr.Image(label = "Pokémon Image"),
        gr.Textbox(label = "True primary type"),
        gr.Textbox(label = "Predicted primary type"),
        gr.Textbox(label = "Primary correct?"),
        gr.Textbox(label = "True secondary type"),
        gr.Textbox(label = "Predicted secondary type"),
        gr.Textbox(label = "Secondary correct?"),
        gr.Textbox(label = "Both types correct?"),
    ],
    title = "Pokémon Type Predictor",
    description = (
        "Select a Pokémon, see its picture, and how our machine learning model "
        "predicts its primary and secondary types."
    )
)

# Launch demo
demo.launch(share = True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://239d7fe0c8d17c9b3f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **IS510 Presenttaion - Battles Prediction**

***By Team 2, Gold Cohort, MSIS 2026***

## **Step 0. Preparation**

In [17]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

# Upload the datasets to my Github and get the links
id_each_team_url = "https://raw.githubusercontent.com/FlalaGoGoGo/IS510_Cases/refs/heads/main/IS510_Pokemon_ID_Each_Team.csv"
single_combats_url = "https://raw.githubusercontent.com/FlalaGoGoGo/IS510_Cases/refs/heads/main/IS510_Pokemon_Single_Combats.csv"
team_combats_url = "https://raw.githubusercontent.com/FlalaGoGoGo/IS510_Cases/refs/heads/main/IS510_Pokemon_Team_Combats.csv"
type_matchup_url = "https://raw.githubusercontent.com/FlalaGoGoGo/IS510_Cases/refs/heads/main/IS510_Pokemon_Type_Matchup_Data.csv"

# Read CSV files into DataFrames
id_each_team_df = pd.read_csv(id_each_team_url)
single_combats_df = pd.read_csv(single_combats_url)
team_combats_df = pd.read_csv(team_combats_url)
type_matchup_df = pd.read_csv(type_matchup_url)

In [18]:
# Get dataset shape
print("ID Each Team: ", id_each_team_df.shape)
print("Single Combats: ", single_combats_df.shape)
print("Team Combats: ", team_combats_df.shape)
print("Type Matchup: ", type_matchup_df.shape)

ID Each Team:  (100, 7)
Single Combats:  (50000, 3)
Team Combats:  (10000, 3)
Type Matchup:  (540, 20)


In [19]:
# Preview training data
print("ID Each Team:")
display(id_each_team_df.head())
print("\nSingle Combats:")
display(single_combats_df.head())
print("\nTeam Combats:")
display(team_combats_df.head())
print("\nType Matchup:")
display(type_matchup_df.head())

ID Each Team:


,#,0,1,2,3,4,5
0,1,132,155,610,382,100,519
1,2,718,357,775,356,123,635
2,3,528,616,293,689,716,688
3,4,73,305,82,383,314,32
4,5,155,753,440,207,482,22



Single Combats:


,First_pokemon,Second_pokemon,Winner
0,266,298,298
1,702,701,701
2,191,668,668
3,237,683,683
4,151,231,151



Team Combats:


,first,second,winner
0,1,1,1
1,1,2,0
2,1,3,1
3,1,4,0
4,1,5,0



Type Matchup:


,Name,Number,Normal,Fire,Water,Electric,Grass,Ice,Fighting,Poison,Ground,Flying,Psychic,Bug,Rock,Ghost,Dragon,Dark,Steel,Fairy
0,Bulbasaur,#001,*1,*2,*0.5,*0.5,*0.25,*2,*0.5,*1,*1,*2,*2,*1,*1,*1,*1,*1,*1,*0.5
1,Ivysaur,#002,*1,*2,*0.5,*0.5,*0.25,*2,*0.5,*1,*1,*2,*2,*1,*1,*1,*1,*1,*1,*0.5
2,Venusaur,#003,*1,*2,*0.5,*0.5,*0.25,*2,*0.5,*1,*1,*2,*2,*1,*1,*1,*1,*1,*1,*0.5
3,Charmander,#004,*1,*0.5,*2,*1,*0.5,*0.5,*1,*1,*2,*1,*1,*0.5,*2,*1,*1,*1,*0.5,*0.5
4,Charmeleon,#005,*1,*0.5,*2,*1,*0.5,*0.5,*1,*1,*2,*1,*1,*0.5,*2,*1,*1,*1,*0.5,*0.5


In [20]:
# Get column names
print("ID Each Team:")
print(id_each_team_df.columns.tolist())
print("\nSingle Combats:")
print(single_combats_df.columns.tolist())
print("\nTeam Combats:")
print(team_combats_df.columns.tolist())
print("\nType Matchup:")
print(type_matchup_df.columns.tolist())

ID Each Team:
['#', '0', '1', '2', '3', '4', '5']

Single Combats:
['First_pokemon', 'Second_pokemon', 'Winner']

Team Combats:
['first', 'second', 'winner']

Type Matchup:
['Name', 'Number', 'Normal', 'Fire', 'Water', 'Electric', 'Grass', 'Ice', 'Fighting', 'Poison', 'Ground', 'Flying', 'Psychic', 'Bug', 'Rock', 'Ghost', 'Dragon', 'Dark', 'Steel', 'Fairy']


In [21]:
# Check missing values in each dataset
print("ID Each Team:")
print(id_each_team_df.isna().sum())
print("\nSingle Combats:")
print(single_combats_df.isna().sum())
print("\nTeam Combats:")
print(team_combats_df.isna().sum())
print("\nType Matchup:")
print(type_matchup_df.isna().sum())

ID Each Team:
#    0
0    0
1    0
2    0
3    0
4    0
5    0
dtype: int64

Single Combats:
First_pokemon     0
Second_pokemon    0
Winner            0
dtype: int64

Team Combats:
first     0
second    0
winner    0
dtype: int64

Type Matchup:
Name        0
Number      0
Normal      0
Fire        0
Water       0
Electric    0
Grass       0
Ice         0
Fighting    0
Poison      0
Ground      0
Flying      0
Psychic     0
Bug         0
Rock        0
Ghost       0
Dragon      0
Dark        0
Steel       0
Fairy       0
dtype: int64


## **Step 1: Prepare Pokémon lookup table (stats + defensive multipliers)**



In [22]:
# Ensure AllPokemons_df has a "dexnum" column
if "#" in AllPokemons_df.columns and "dexnum" not in AllPokemons_df.columns:
    AllPokemons_df = AllPokemons_df.rename(columns = {"#": "dexnum"})

# Make sure dexnum in AllPokemons_df is numeric
AllPokemons_df["dexnum"] = pd.to_numeric(AllPokemons_df["dexnum"], errors = "coerce")

# Ensure type_matchup_df uses "dexnum" as the ID column
if "Number" in type_matchup_df.columns and "dexnum" not in type_matchup_df.columns:
    type_matchup_df = type_matchup_df.rename(columns = {"Number": "dexnum"})

# Clean the "dexnum" column in type_matchup_df
def clean_dexnum(val):

    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    # Remove leading "#" if present
    if s.startswith("#"):
        s = s[1:]
    # Remove any leading/trailing spaces again just in case
    s = s.strip()
    # Try to convert to integer
    try:
        return int(s)
    except ValueError:
        return np.nan

type_matchup_df["dexnum"] = type_matchup_df["dexnum"].apply(clean_dexnum)

# Drop rows where dexnum could not be parsed
type_matchup_df = type_matchup_df.dropna(subset=["dexnum"])

# Define ID columns and multiplier columns for the matchup table
id_cols_in_matchup = []
if "dexnum" in type_matchup_df.columns:
    id_cols_in_matchup.append("dexnum")
if "Name" in type_matchup_df.columns:
    id_cols_in_matchup.append("Name")

multiplier_cols = [c for c in type_matchup_df.columns if c not in id_cols_in_matchup]

print("ID columns in type_matchup_df:", id_cols_in_matchup)
print("Multiplier columns (attack types):", multiplier_cols)

# Convert multiplier strings (e.g., "*2", "*0.5") to float values
def parse_multiplier(val):
    if pd.isna(val):
        return 1.0
    if isinstance(val, str):
        s = val.strip()
        if s.startswith("*"):
            s = s[1:]
        try:
            return float(s)
        except ValueError:
            return 1.0
    try:
        return float(val)
    except ValueError:
        return 1.0

type_matchup_numeric_df = type_matchup_df.copy()
for col in multiplier_cols:
    type_matchup_numeric_df[col] = type_matchup_numeric_df[col].apply(parse_multiplier)

# Select core columns from AllPokemons_df
stat_cols = ["hp", "attack", "defense", "sp_atk", "sp_def", "speed", "total"]
pokemon_base_cols = ["dexnum", "name", "type1", "type2"] + stat_cols
pokemon_base_df = AllPokemons_df[pokemon_base_cols].copy()

# Make sure "dexnum" has the same dtype (int) in both tables before merging
pokemon_base_df["dexnum"] = pd.to_numeric(pokemon_base_df["dexnum"], errors="coerce")
pokemon_base_df = pokemon_base_df[
    pokemon_base_df["dexnum"].notna() & np.isfinite(pokemon_base_df["dexnum"])
].copy()
pokemon_base_df["dexnum"] = pokemon_base_df["dexnum"].astype(int)

type_matchup_numeric_df = type_matchup_numeric_df[
    type_matchup_numeric_df["dexnum"].notna() & np.isfinite(type_matchup_numeric_df["dexnum"])
].copy()
type_matchup_numeric_df["dexnum"] = type_matchup_numeric_df["dexnum"].astype(int)

print("pokemon_base_df dexnum dtype:", pokemon_base_df["dexnum"].dtype)
print("type_matchup_numeric_df dexnum dtype:", type_matchup_numeric_df["dexnum"].dtype)

# Merge base stats/types with numeric matchup multipliers on "dexnum"
pokemon_lookup_df = pokemon_base_df.merge(
    type_matchup_numeric_df[["dexnum"] + multiplier_cols],
    on = "dexnum",
    how = "inner"
)

print("pokemon_lookup_df shape:", pokemon_lookup_df.shape)
display(pokemon_lookup_df.head())

ID columns in type_matchup_df: ['dexnum', 'Name']
Multiplier columns (attack types): ['Normal', 'Fire', 'Water', 'Electric', 'Grass', 'Ice', 'Fighting', 'Poison', 'Ground', 'Flying', 'Psychic', 'Bug', 'Rock', 'Ghost', 'Dragon', 'Dark', 'Steel', 'Fairy']
pokemon_base_df dexnum dtype: int64
type_matchup_numeric_df dexnum dtype: int64
pokemon_lookup_df shape: (540, 29)


,dexnum,name,type1,type2,hp,attack,defense,sp_atk,sp_def,speed,...,Ground,Flying,Psychic,Bug,Rock,Ghost,Dragon,Dark,Steel,Fairy
0,1,Bulbasaur,Grass,Poison,45.0,49.0,49.0,65.0,65.0,45.0,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
1,2,Ivysaur,Grass,Poison,60.0,62.0,63.0,80.0,80.0,60.0,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
2,3,Venusaur,Grass,Poison,80.0,82.0,83.0,100.0,100.0,80.0,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
3,4,Charmander,Fire,NaN,39.0,52.0,43.0,60.0,50.0,65.0,...,2.0,1.0,1.0,0.5,2.0,1.0,1.0,1.0,0.5,0.5
4,5,Charmeleon,Fire,NaN,58.0,64.0,58.0,80.0,65.0,80.0,...,2.0,1.0,1.0,0.5,2.0,1.0,1.0,1.0,0.5,0.5


## **Step 2: Feature engineering for single combats (stats differences + type advantage)**

In [23]:
# Merge single combat data with Pokémon lookup table for First and Second Pokémon
# Assumes single_combats_df has columns: "First_pokemon", "Second_pokemon", "Winner"
single_enriched_df = single_combats_df.merge(
    pokemon_lookup_df.add_prefix("A_"),
    left_on = "First_pokemon",
    right_on = "A_dexnum",
    how = "inner"
).merge(
    pokemon_lookup_df.add_prefix("B_"),
    left_on = "Second_pokemon",
    right_on = "B_dexnum",
    how = "inner"
)

print("single_enriched_df shape:", single_enriched_df.shape)
display(single_enriched_df.head())

# Compute stat difference features: A - B
for col in stat_cols:
    single_enriched_df[f"{col}_diff"] = single_enriched_df[f"A_{col}"] - single_enriched_df[f"B_{col}"]

# Helper function to compute type advantage: attacker hits defender
def compute_type_advantage_row(attacker_row, defender_row, attacker_prefix, defender_prefix, multiplier_columns):
    atk_types = [attacker_row[f"{attacker_prefix}type1"]]
    if attacker_row[f"{attacker_prefix}type2"] != "None":
        atk_types.append(attacker_row[f"{attacker_prefix}type2"])

    multipliers = []
    for t in atk_types:
        if t in multiplier_columns:
            col_name = f"{defender_prefix}{t}"
            if col_name in defender_row.index:
                multipliers.append(defender_row[col_name])

    if len(multipliers) == 0:
        return 1.0
    return max(multipliers)

# Compute type advantage features: A attacking B, and B attacking A
single_enriched_df["A_type_advantage"] = single_enriched_df.apply(
    lambda row: compute_type_advantage_row(row, row, "A_", "B_", multiplier_cols),
    axis = 1
)

single_enriched_df["B_type_advantage"] = single_enriched_df.apply(
    lambda row: compute_type_advantage_row(row, row, "B_", "A_", multiplier_cols),
    axis = 1
)

single_enriched_df["type_advantage_diff"] = (
    single_enriched_df["A_type_advantage"] - single_enriched_df["B_type_advantage"]
)

# Define label: 1 if First_pokemon wins, 0 otherwise
# Assumes "Winner" column stores the dex number of the winning Pokémon
single_enriched_df["First_wins"] = (
    single_enriched_df["Winner"] == single_enriched_df["First_pokemon"]
).astype(int)

print("\nLabel distribution (First_wins):")
print(single_enriched_df["First_wins"].value_counts(normalize = True))

# Show a few engineered feature columns
engineered_cols_preview = [f"{c}_diff" for c in stat_cols] + [
    "A_type_advantage", "B_type_advantage", "type_advantage_diff", "First_wins"
]
display(single_enriched_df[engineered_cols_preview].head())

single_enriched_df shape: (15819, 61)


,First_pokemon,Second_pokemon,Winner,A_dexnum,A_name,A_type1,A_type2,A_hp,A_attack,A_defense,...,B_Ground,B_Flying,B_Psychic,B_Bug,B_Rock,B_Ghost,B_Dragon,B_Dark,B_Steel,B_Fairy
0,702,701,701,702,Dedenne,Electric,Fairy,67.0,58.0,57.0,...,0.0,2.0,2.0,0.25,1.0,1.0,1.0,0.5,1.0,2.0
1,237,683,683,237,Hitmontop,Fighting,NaN,50.0,95.0,95.0,...,1.0,1.0,1.0,0.50,1.0,1.0,0.0,0.5,2.0,1.0
2,73,545,545,73,Tentacruel,Water,Poison,80.0,70.0,65.0,...,1.0,2.0,2.0,0.50,2.0,1.0,1.0,1.0,1.0,0.5
3,220,763,763,220,Swinub,Ice,Ground,50.0,50.0,40.0,...,0.5,2.0,1.0,2.00,1.0,1.0,1.0,1.0,1.0,1.0
4,701,624,701,701,Hawlucha,Fighting,Flying,78.0,92.0,75.0,...,2.0,0.5,0.0,1.00,0.5,0.5,0.5,0.5,0.5,1.0



Label distribution (First_wins):
First_wins
0    0.521651
1    0.478349
Name: proportion, dtype: float64


,hp_diff,attack_diff,defense_diff,sp_atk_diff,sp_def_diff,speed_diff,total_diff,A_type_advantage,B_type_advantage,type_advantage_diff,First_wins
0,-11.0,-34.0,-18.0,7.0,4.0,-17.0,-69.0,2.0,0.5,1.5,0
1,-51.0,23.0,23.0,-64.0,21.0,41.0,-7.0,0.5,2.0,-1.5,0
2,20.0,-30.0,-24.0,25.0,51.0,-12.0,30.0,1.0,0.5,0.5,0
3,-22.0,-70.0,-58.0,-20.0,-68.0,-22.0,-260.0,2.0,2.0,0.0,0
4,33.0,7.0,5.0,34.0,23.0,58.0,160.0,4.0,1.0,3.0,1


## **Step 3: Define feature sets and train–test split**

In [24]:
# Stats-only feature columns (differences)
stats_diff_features = [f"{c}_diff" for c in stat_cols]

# Stats + type-advantage feature columns
type_features = ["A_type_advantage", "B_type_advantage", "type_advantage_diff"]
all_features = stats_diff_features + type_features

# Define X and y
X_all = single_enriched_df[all_features].copy()
y = single_enriched_df["First_wins"].copy()

# Train–test split
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

# Separate stats-only feature matrices (for baseline model)
X_train_stats = X_train_all[stats_diff_features].copy()
X_test_stats = X_test_all[stats_diff_features].copy()

print("X_train_all shape:", X_train_all.shape)
print("X_test_all shape: ", X_test_all.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True))

X_train_all shape: (12655, 10)
X_test_all shape:  (3164, 10)
y_train distribution:
First_wins
0    0.521612
1    0.478388
Name: proportion, dtype: float64


## **Step 4: Baseline model - Logistic Regression (stats-only features)**

In [25]:
# Build a pipeline: StandardScaler + LogisticRegression
logreg_stats_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=  1000, random_state = 42))
])

# Fit on stats-only features
logreg_stats_clf.fit(X_train_stats, y_train)

# Predict on test set
y_pred_stats = logreg_stats_clf.predict(X_test_stats)
y_proba_stats = logreg_stats_clf.predict_proba(X_test_stats)[:, 1]

# Evaluate
acc_stats = accuracy_score(y_test, y_pred_stats)
auc_stats = roc_auc_score(y_test, y_proba_stats)

print("Logistic Regression (stats-only features):")
print(f"-- Accuracy: {acc_stats:.4f}")
print(f"-- ROC AUC:  {auc_stats:.4f}")
print("\nClassification report (stats-only):")
print(classification_report(y_test, y_pred_stats, digits=3))

Logistic Regression (stats-only features):
-- Accuracy: 0.5544
-- ROC AUC:  0.5640

Classification report (stats-only):
              precision    recall  f1-score   support

           0      0.551     0.788     0.649      1651
           1      0.564     0.299     0.391      1513

    accuracy                          0.554      3164
   macro avg      0.558     0.544     0.520      3164
weighted avg      0.557     0.554     0.525      3164



## **Step 5: Extended models - Logistic Regression (all features) and Random Forest**

In [26]:
# Logistic Regression with all features (stats + type advantage)
logreg_all_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

logreg_all_clf.fit(X_train_all, y_train)

y_pred_all = logreg_all_clf.predict(X_test_all)
y_proba_all = logreg_all_clf.predict_proba(X_test_all)[:, 1]

acc_all = accuracy_score(y_test, y_pred_all)
auc_all = roc_auc_score(y_test, y_proba_all)

print("Logistic Regression (stats + type advantage):")
print(f"-- Accuracy: {acc_all:.4f}")
print(f"-- ROC AUC: {auc_all:.4f}")
print("\nClassification report (all features):")
print(classification_report(y_test, y_pred_all, digits=3))

# Random Forest with all features
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf_clf.fit(X_train_all, y_train)

y_pred_rf = rf_clf.predict(X_test_all)
y_proba_rf = rf_clf.predict_proba(X_test_all)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_proba_rf)

print("\nRandom Forest (all features):")
print(f"  Accuracy: {acc_rf:.4f}")
print(f"  ROC AUC:  {auc_rf:.4f}")
print("\nClassification report (Random Forest):")
print(classification_report(y_test, y_pred_rf, digits=3))

Logistic Regression (stats + type advantage):
-- Accuracy: 0.5550
-- ROC AUC: 0.5606

Classification report (all features):
              precision    recall  f1-score   support

           0      0.553     0.772     0.644      1651
           1      0.561     0.319     0.406      1513

    accuracy                          0.555      3164
   macro avg      0.557     0.545     0.525      3164
weighted avg      0.557     0.555     0.530      3164


Random Forest (all features):
  Accuracy: 0.5970
  ROC AUC:  0.6397

Classification report (Random Forest):
              precision    recall  f1-score   support

           0      0.600     0.682     0.639      1651
           1      0.592     0.504     0.545      1513

    accuracy                          0.597      3164
   macro avg      0.596     0.593     0.592      3164
weighted avg      0.596     0.597     0.594      3164



## **Step 6: Extended models - Logistic Regression (all features) and Random Forest**

In [27]:
model_results = {
    "LogReg_Stats": {"accuracy": acc_stats, "auc": auc_stats},
    "LogReg_All": {"accuracy": acc_all, "auc": auc_all},
    "RandomForest": {"accuracy": acc_rf, "auc": auc_rf}
}

print("Summary of model performance (Accuracy, ROC AUC):")
for name, res in model_results.items():
    print(f"{name}: Accuracy = {res['accuracy']:.4f}, ROC AUC = {res['auc']:.4f}")

Summary of model performance (Accuracy, ROC AUC):
LogReg_Stats: Accuracy = 0.5544, ROC AUC = 0.5640
LogReg_All: Accuracy = 0.5550, ROC AUC = 0.5606
RandomForest: Accuracy = 0.5970, ROC AUC = 0.6397


In [28]:
# Select Random Forest as the final battle prediction model.
battle_model = rf_clf
battle_feature_cols = all_features
battle_stats_diff_features = stats_diff_features

print("Final model selected for battle prediction: Random Forest (all features)")
print(f"Final test accuracy: {acc_rf:.4f}")
print(f"Final test ROC AUC:  {auc_rf:.4f}")

Final model selected for battle prediction: Random Forest (all features)
Final test accuracy: 0.5970
Final test ROC AUC:  0.6397


## **Step 7: Helper function to build features for a custom Pokémon pair**

In [29]:
def build_pair_features_from_rows(row_a, row_b, stat_columns, multiplier_columns):
    # Stat differences: A - B
    features = {}
    for col in stat_columns:
        features[f"{col}_diff"] = row_a[col] - row_b[col]

    # Type advantage helper
    def type_advantage(attacker, defender):

        atk_types = [attacker["type1"]]

        # Some Pokémon have no second type (NaN). We only add it if it exists.
        if pd.notna(attacker["type2"]):
            atk_types.append(attacker["type2"])

        multipliers = []
        for t in atk_types:
            if t in multiplier_columns:
                multipliers.append(defender[t])

        if len(multipliers) == 0:
            return 1.0
        return max(multipliers)

    # Compute type advantages A→B and B→A
    a_adv = type_advantage(row_a, row_b)
    b_adv = type_advantage(row_b, row_a)

    features["A_type_advantage"] = a_adv
    features["B_type_advantage"] = b_adv
    features["type_advantage_diff"] = a_adv - b_adv

    # Build DataFrame in the correct feature column order
    feature_row_df = pd.DataFrame([features])[battle_feature_cols]
    return feature_row_df

# Quick sanity check with the first two Pokémon in pokemon_lookup_df
test_row_a = pokemon_lookup_df.iloc[0]
test_row_b = pokemon_lookup_df.iloc[1]
test_features_df = build_pair_features_from_rows(
    test_row_a, test_row_b, stat_cols, multiplier_cols
)

print("Example feature row for a test battle (A vs B):")
display(test_features_df)

Example feature row for a test battle (A vs B):


,hp_diff,attack_diff,defense_diff,sp_atk_diff,sp_def_diff,speed_diff,total_diff,A_type_advantage,B_type_advantage,type_advantage_diff
0,-15.0,-13.0,-14.0,-15.0,-15.0,-15.0,-87.0,1.0,1.0,0.0


## **Step 8: Gradio interactive demo – choose two Pokémon and predict the winner**

In [30]:
import gradio as gr
import requests
from PIL import Image
from io import BytesIO

# Helper function to download image from URL and return a PIL image
def fetch_image(url):

    if not isinstance(url, str) or url.strip() == "":
        return None
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        return Image.open(BytesIO(resp.content))
    except Exception:
        return None

# Build a mapping from Pokémon name to image URL using AllPokemons_df
name_to_url_battle = dict(zip(AllPokemons_df["name"], AllPokemons_df["url"]))

# Build dropdown options like "001 - Bulbasaur", sorted by dexnum
pokemon_df_sorted = pokemon_lookup_df.copy()
pokemon_df_sorted = pokemon_df_sorted[pokemon_df_sorted["dexnum"].notna()].copy()
pokemon_df_sorted["dexnum"] = pokemon_df_sorted["dexnum"].astype(int)
pokemon_df_sorted = pokemon_df_sorted.sort_values("dexnum")

pokemon_options = []
for _, row in pokemon_df_sorted.iterrows():
    num = int(row["dexnum"])
    name = row["name"]
    label = f"{num:03d} - {name}"   # e.g., "001 - Bulbasaur"
    pokemon_options.append(label)


def predict_battle_from_dropdown(option_a, option_b):
    # Basic input validation
    if (option_a is None or option_a == "") or (option_b is None or option_b == ""):
        return (
            None,  # Image A
            "Please select Pokémon A.",
            None,  # Image B
            "Please select Pokémon B.",
            "N/A",
            "N/A",
            "N/A"
        )

    # Parse labels "001 - Bulbasaur" -> dex number and name
    try:
        dex_a_str, name_a = option_a.split(" - ", 1)
        dex_b_str, name_b = option_b.split(" - ", 1)
    except ValueError:
        return (
            None,
            "Invalid selection for Pokémon A.",
            None,
            "Invalid selection for Pokémon B.",
            "N/A",
            "N/A",
            "N/A"
        )

    dex_a = int(dex_a_str)
    dex_b = int(dex_b_str)

    # Look up rows in pokemon_lookup_df
    row_a = pokemon_lookup_df[pokemon_lookup_df["dexnum"] == dex_a]
    row_b = pokemon_lookup_df[pokemon_lookup_df["dexnum"] == dex_b]

    if row_a.empty or row_b.empty:
        return (
            None,
            f"Pokémon A not found for dex {dex_a}.",
            None,
            f"Pokémon B not found for dex {dex_b}.",
            "N/A",
            "N/A",
            "N/A"
        )

    row_a = row_a.iloc[0]
    row_b = row_b.iloc[0]

    # Model prediction
    # Build feature row and predict probability that A (First) wins
    X_pair = build_pair_features_from_rows(row_a, row_b, stat_cols, multiplier_cols)
    proba = battle_model.predict_proba(X_pair)[0, 1]  # P(First wins)

    p_a = proba
    p_b = 1.0 - proba

    predicted_winner_name = name_a if p_a >= 0.5 else name_b
    predicted_winner_side = "A" if p_a >= 0.5 else "B"

    # Prepare display strings for both Pokémon
    info_a = f"{name_a} (dex {dex_a})\nTypes: {row_a['type1']}"
    if pd.notna(row_a["type2"]):
        info_a += f" / {row_a['type2']}"
    info_a += f"\nTotal base stats: {row_a['total']}"

    info_b = f"{name_b} (dex {dex_b})\nTypes: {row_b['type1']}"
    if pd.notna(row_b["type2"]):
        info_b += f" / {row_b['type2']}"
    info_b += f"\nTotal base stats: {row_b['total']}"

    winner_text = f"{predicted_winner_name}"
    proba_text = f"P(A wins) = {p_a:.3f}\nP(B wins) = {p_b:.3f}"

    # Fetch images from AllPokemons_df via name_to_url_battle
    img_a_url = name_to_url_battle.get(name_a, None)
    img_b_url = name_to_url_battle.get(name_b, None)
    img_a = fetch_image(img_a_url)
    img_b = fetch_image(img_b_url)

    # Historical win rates from the dataset
    # We use single_combats_df to see how many times A or B actually won
    # in recorded 1v1 battles (in both orders: A vs B and B vs A).
    df_ab = single_combats_df[
        (single_combats_df["First_pokemon"] == dex_a) &
        (single_combats_df["Second_pokemon"] == dex_b)
    ]
    df_ba = single_combats_df[
        (single_combats_df["First_pokemon"] == dex_b) &
        (single_combats_df["Second_pokemon"] == dex_a)
    ]

    total_battles = len(df_ab) + len(df_ba)

    if total_battles == 0:
        history_text = (
            "No battles between these two Pokémon in the dataset.\n"
            "So we cannot check whether the model is correct here."
        )
        return img_a, info_a, img_b, info_b, winner_text, proba_text, history_text

    # Count wins for A and B
    wins_a = 0
    wins_b = 0

    # For A as First, B as Second
    if not df_ab.empty:
        wins_a += (df_ab["Winner"] == dex_a).sum()
        wins_b += (df_ab["Winner"] == dex_b).sum()

    # For B as First, A as Second
    if not df_ba.empty:
        wins_a += (df_ba["Winner"] == dex_a).sum()
        wins_b += (df_ba["Winner"] == dex_b).sum()

    rate_a = wins_a / total_battles
    rate_b = wins_b / total_battles

    # Decide historical majority winner
    if wins_a > wins_b:
        majority_side = "A"
        majority_name = name_a
    elif wins_b > wins_a:
        majority_side = "B"
        majority_name = name_b
    else:
        majority_side = "tie"
        majority_name = "tie"

    # Check if the model prediction matches historical majority
    if total_battles < 5 or majority_side == "tie":
        correctness = (
            f"Total battles in dataset: {total_battles}\n"
            f"{name_a} wins: {wins_a}\n"
            f"{name_b} wins: {wins_b}\n\n"
            "The matchup is either perfectly balanced or we do not have enough battles,\n"
            "so we do not label the model as 'correct' or 'incorrect' here."
        )
    else:
        is_correct = (predicted_winner_side == majority_side)
        correctness = (
            f"Total battles in dataset: {total_battles}\n"
            f"{name_a} wins: {wins_a}\n"
            f"{name_b} wins: {wins_b}\n\n"
            f"Historical majority winner: {majority_name}\n"
            f"Model prediction matches majority? "
            f"{'Yes ✅' if is_correct else 'No ❌'}"
        )

    # Return images + text outputs
    return img_a, info_a, img_b, info_b, winner_text, proba_text, correctness


# Define Gradio interface
battle_demo = gr.Interface(
    fn = predict_battle_from_dropdown,
    inputs = [
        gr.Dropdown(choices = pokemon_options, label = "Choose Pokémon A (First)"),
        gr.Dropdown(choices = pokemon_options, label = "Choose Pokémon B (Second)")
    ],
    outputs = [
        gr.Image(label = "Pokémon A Image"),
        gr.Textbox(label = "Pokémon A Info", lines = 4),
        gr.Image(label = "Pokémon B Image"),
        gr.Textbox(label = "Pokémon B Info", lines = 4),
        gr.Textbox(label = "Predicted Winner", lines = 2),
        gr.Textbox(label = "Win Probabilities", lines = 2),
        gr.Textbox(label = "Historical Data", lines = 6),
    ],
    title = "Pokémon 1v1 Battle Predictor",
    description = (
        "Made by Team 2, Gold Cohort, MSIS 2026. "
        "Select two Pokémon and see how our Random Forest model predicts the winner.\n"
        "We also show historical win rates from the dataset and whether the model "
        "agrees with the majority outcome."
    )
)

# Launch demo
battle_demo.launch(share = True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8c2277cef95f93da4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
